In [ ]:
import sys 
sys.path.append('/Users/amalyajohnson/Desktop/SpectraScope/spectrascope_local/')
%load_ext autoreload
%autoreload 2

from spectrascope.SpectraScope import *
from spectrascope.features import * 
from sklearn.linear_model import Ridge, Lasso, ElasticNet
import SPA 
import os
from mat4py import loadmat

In [ ]:
spec_dir = '/Users/amalyajohnson/Desktop/SpectraScope/Datasets/Grape-Chemical-Composition-main/Spectroscopic data (Matlab files)/'
true_dir = '/Users/amalyajohnson/Desktop/SpectraScope/Datasets/Grape-Chemical-Composition-main/Ground Truth/'

In [ ]:
data = loadmat(spec_dir + 'specData.mat')
XSC = np.array(data['XSC'])
XSF = np.array(data['XSF'])

data = loadmat(spec_dir + 'ramanData.mat')
XRC = np.array(data['XRC'])
XRF = np.array(data['XRF'])

data = loadmat(true_dir + 'trueData.mat')
YF = np.array(data['YF'])
YC = np.array(data['YC']) 

In [ ]:
from mat4py import loadmat

fn = spec_dir + 'RAMAN_Data_09082022.mat'
fn2 = spec_dir + 'NIR_Data_09082022.mat'

wls_nir = []
wls_rm = []
shifts = []
names_nir = []
names_rm = []
data_nir = {}
data_rm = {}
for fn in os.listdir(spec_dir):
    if 'Data.mat' not in fn: 
        data = loadmat(spec_dir + fn)
        name = fn.split('/')[-1].split('.')[0]
        wl = np.array(data[name]['wvlength'])
    else: 
        continue
    if 'NIR' in name:
        names_nir.append(name)
        wls_nir.append(wl)
        for key2 in data[name]['spec']: 
            arr = data[name]['spec'][key2]
            arr = np.array(arr)
            data_nir[name + '_' + key2] = arr
    elif 'RAMAN' in name: 
        shift = data[name]['raman_shift']
        shifts.append(shift)
        names_rm.append(name) 
        wls_rm.append(wl)
        for key1 in data[name]['raman']: 
            for key2 in data[name]['raman'][key1]: 
                arr = data[name]['raman'][key1][key2]
                arr = np.array(arr)
                if key1 == 'Cab':
                    data_rm[name + '_' + key1 + '_' + key2] = arr 

In [ ]:
shiftsi = np.array([(int(i), s[0]) for i,s in enumerate(shifts[0]) if str(s[0]) != 'nan'])
shifts_ids = shiftsi[:,0]
raman_shifts = shiftsi[:,1]

wlsi = np.array([(int(i), s[0]) for i,s in enumerate(wls_rm[0]) if str(s[0]) != 'nan'])
wls_ids = wlsi[:,0]
raman_wls = wlsi[:,1]

nir_wls = np.array(wls_nir[0])
nir_wls = nir_wls.reshape(len(nir_wls),)

In [ ]:
XRC = XRC[:,np.array([int(i) for i in wls_ids])]
XRF = XRF[:,np.array([int(i) for i in wls_ids])]

In [ ]:
fig = make_subplots(1,2)
fig.add_trace(go.Scatter(x=nir_wls, y=XSC[0]), row=1, col=1)
fig.add_trace(go.Scatter(x = raman_wls, y = XRC[0]), row=1, col=2)

fig.add_trace(go.Scatter(x=nir_wls, y=XSF[0]), row=1, col=1)
fig.add_trace(go.Scatter(x = raman_wls, y = XRF[0]), row=1, col=2)

fig.show()

In [ ]:
# Deprecated ; previous loading data 
shifts = np.array(shifts)

Fph = white['pH'].to_numpy()[white['pH'].isna().to_numpy() == False]
FBrix = white['Brix'].to_numpy()[white['Brix'].isna().to_numpy() == False]

Cph = red['pH'].to_numpy()[red['pH'].isna().to_numpy() == False]
CBrix = red['Brix'].to_numpy()[red['pH'].isna().to_numpy() == False]

XC_nir = data_nir['NIR_Data_07292022_Cab']
XF_nir = data_nir['NIR_Data_07292022_French'][:-1][[white['pH'].isna().to_numpy() == False][0][:148]]

XC_rm = data_rm['RAMAN_Data_07292022_Cab1_blr']
XF_rm = data_rm['RAMAN_Data_07292022_French_blr'][[white['pH'].isna().to_numpy() == False][0][:148]]

wlsi = np.array([(int(i), s[0]) for i,s in enumerate(wls[0]) if str(s[0]) != 'nan'])
ids = wlsi[:,0]
wlsi = wlsi[:,1]
XF_nir = XF_nir[:,np.array([int(i) for i in ids])]

In [ ]:
ph = YF[:,0]
ids_keep = np.argwhere(np.isnan(ph) == False).ravel()
ph = ph[ids_keep]
XSF = XSF[ids_keep]
XRF = XRF[ids_keep]
XSC = XSC[ids_keep]
XRC = XRC[ids_keep]


XF = [XSF, XRF]
ss_f = SpectraScope(XF, ph, specnames=['nir', 'rm'], yname='ph', random_seed=10)
ss_f.add_x_axis('rm', raman_wls)
ss_f.add_x_axis('nir', nir_wls)

In [ ]:
ss_f.add_spectral_feature('rm', 'pointwise_features', xrange=ss_f.rm_xaxis)
ss_f.add_spectral_feature('rm', 'polynomial_featuresall')
ss_f.add_spectral_feature('rm', 'featureagg', n_clusters=15)
ss_f.add_spectral_feature('rm', 'pca', n_components=15)
ss_f.add_spectral_feature('rm', 'cdf')
ss_f.add_spectral_feature('rm', 'cwt')

ss_f.add_spectral_feature('nir', 'pointwise_features', xrange=ss_f.nir_xaxis)
ss_f.add_spectral_feature('nir', 'polynomial_featuresall')
ss_f.add_spectral_feature('nir', 'featureagg', n_clusters=15)
ss_f.add_spectral_feature('nir', 'pca', n_components=15)
ss_f.add_spectral_feature('nir', 'cdf')
ss_f.add_spectral_feature('nir', 'cwt')

In [ ]:
ss_f.transform_feature('rm_pointwise_features', degree=2, interaction=False)
ss_f.transform_feature('nir_pointwise_features', degree=2, interaction=False)

In [ ]:
ss_f.combine_feature('rm_pointwise_features', 'nir_pointwise_features')
# ss_f.transform_feature('rm_nir_pointwise_features', degree=1, interaction=True)

In [ ]:
RF_kwargs = {'estimator':RandomForestRegressor(), 
                'param_grid':{'n_estimators':[200], 
                                        'criterion':['squared_error'],
                                        'max_depth': [50, None]},
                'refit':True,
                'metric':mean_squared_error}

lcen_kwargs = {'l1_ratio':[0.001,0.005, 0.01, 0.1,0.3], 
               'K_fold':3, 
               'cv_method':'Kfold', 
               'alpha':5, 
               'degree':[1,2],
               'trans_type':'poly', 
               'LCEN_cutoff':5e-2, 
               'LCEN_interaction':True, 
               'scale_X':True, 
               'scale_y':True}


Lasso_kwargs = {'estimator':Lasso()}

In [ ]:
feats = []
for spec in ss_f.specnames:
    attr_names = [feat for feat in dir(ss_f) if '' in feat and f'{spec}_' in feat and type(getattr(ss_f, feat)) == np.ndarray and 'names' not in feat and 'test' not in feat and 'train' not in feat and 'xaxis' not in feat]
    for attr_name in attr_names: 
        # print(attr_name, getattr(ss_f, attr_name).shape)
        feats.append(attr_name)
        # ss_f.do_sk(attr_name, 'ph', 0.25, use_pipeline=False)

In [189]:
ss_f.do_sk('rm_pointwise_features', 'ph', 0.25, use_pipeline=False, estimator=RandomForestRegressor())
ss_f.do_sk('rm_pca', 'ph', 0.25, use_pipeline=False, estimator=RandomForestRegressor())
ss_f.do_sk('rm_polynomial_featuresall', 'ph', 0.25, use_pipeline=False, estimator=RandomForestRegressor())
ss_f.do_sk('rm_featureagg', 'ph', 0.25, use_pipeline=False, estimator=RandomForestRegressor())
ss_f.do_sk('rm_cdf', 'ph', 0.25, use_pipeline=False, estimator=RandomForestRegressor())

Model is not fitted yet.
Model is not fitted yet.
Model is not fitted yet.
Model is not fitted yet.


In [219]:
ss_f.do_sk('nir_pointwise_features', 'ph', 0.25, use_pipeline=False, estimator=RandomForestRegressor())
ss_f.do_sk('nir_pca', 'ph', 0.25, use_pipeline=False, estimator=RandomForestRegressor())
ss_f.do_sk('nir_polynomial_featuresall', 'ph', 0.25, use_pipeline=False, estimator=RandomForestRegressor())
ss_f.do_sk('nir_featureagg', 'ph', 0.25, use_pipeline=False, estimator=RandomForestRegressor())
ss_f.do_sk('nir_cdf', 'ph', 0.25, use_pipeline=False, estimator=RandomForestRegressor())

Model is not fitted yet.


KeyboardInterrupt: 

In [198]:
ss_f.plot_df_heatmap(pd.DataFrame(ss_f.best_fits).T)

In [218]:
from spectrascope.plot_spectral_features import plot_model as pm 
from spectrascope.plot_spectral_features import plot_spectral_features as psf 

fig = make_subplots(1,2, specs=[[{"secondary_y": True}, {"secondary_y": True}]], 
                    subplot_titles=['pointwise_features', 'cdf'])#, specs=[[{"secondary_y": True}, {"secondary_y": True}]], subplot_titles=['RF Feature Importances', 'LCEN Selected Features'])

for i, feat in enumerate(['pointwise_features', 'cdf']):
    model = getattr(ss_f, f'rm_{feat}_ph_RandomForestRegressor')

    selected_features = model['selected_features']
    selected_vals = model['selected_vals']
    raw = getattr(ss_f, f'rm_{feat}')
    xaxis = getattr(ss_f, f'rm_xaxis')

        
    # feature_importances = model['P'][0].best_estimator_.feature_importances_

    feature_importances = model['P'].feature_importances_
    feature_importances = feature_importances / np.max(feature_importances)
    plotter = pm(selected_features=feature_importances, raw=ss_f.rm_pointwise_features[0].ravel(), fig=fig, xaxis=xaxis,row=1,col=i+1, selected_vals=selected_vals, plot_importances=True)
    plotter.plot_pointwise_features()

fig.show()

In [286]:
counter = 0
for i, val in enumerate(ph):
    for counter in range(6):
        # print(i,counter)
        if ph[i + counter] == val: 
            print(i, i+counter)
        else:
            print('\n')
    print('\n')
        

0 0
0 1
0 2
0 3
0 4
0 5
\m
1 1
1 2
1 3
1 4
1 5


\m
2 2
2 3
2 4
2 5




\m
3 3
3 4
3 5






\m
4 4
4 5








\m
5 5










\m
6 6
6 7
6 8
6 9
6 10
6 11
\m
7 7
7 8
7 9
7 10
7 11


\m
8 8
8 9
8 10
8 11




\m
9 9
9 10
9 11






\m
10 10
10 11








\m
11 11










\m
12 12
12 13
12 14
12 15
12 16
12 17
\m
13 13
13 14
13 15
13 16
13 17
13 18
\m
14 14
14 15
14 16
14 17
14 18
14 19
\m
15 15
15 16
15 17
15 18
15 19
15 20
\m
16 16
16 17
16 18
16 19
16 20
16 21
\m
17 17
17 18
17 19
17 20
17 21
17 22
\m
18 18
18 19
18 20
18 21
18 22
18 23
\m
19 19
19 20
19 21
19 22
19 23


\m
20 20
20 21
20 22
20 23




\m
21 21
21 22
21 23






\m
22 22
22 23








\m
23 23










\m
24 24
24 25
24 26
24 27
24 28


\m
25 25
25 26
25 27
25 28




\m
26 26
26 27
26 28






\m
27 27
27 28








\m
28 28










\m
29 29
29 30
29 31
29 32
29 33
29 34
\m
30 30
30 31
30 32
30 33
30 34


\m
31 31
31 32
31 33
31 34




\m
32 32
32 33
32 34






\m
33 33
33 34








\m
34 34










\m
35 35

IndexError: index 1037 is out of bounds for axis 0 with size 1037

In [ ]:
ss_f.do_all(feats, 'ph', 0.25, sk_kwargs=[RF_kwargs], do_all_existing=True)

In [ ]:
ss_f.do_sk('rm_pointwise_features', 'ph', 0.25, use_pipeline=False, **RF_kwargs)
ss_f.do_sk('rm_nir_pointwise_features', 'ph', 0.25, use_pipeline=False, **RF_kwargs)
ss_f.do_sk('rm_nir_pointwise_features_trans', 'ph', 0.25, use_pipeline=False, **RF_kwargs)
#### ss_f.do_sk('rm_pointwise_features_trans', 'ph', 0.25, **RF_kwargs)
ss_f.do_sk('rm_polynomial_featuresall', 'ph', 0.25, use_pipeline=False, **RF_kwargs)

In [ ]:
ss_f.do_LCEN('rm_pointwise_features', 'ph', 0.25, **lcen_kwargs)

In [ ]:
ss_f.rm_pointwise_features_ph_LCEN['selected_features']

In [ ]:
plt.plot(ss_f.rm_pointwise_features_ph_LCEN['yhat_test_nontrans'])

In [ ]:
ss_f.do_sk('rm_cdf', 'ph', 0.25, **RF_kwargs)
ss_f.do_sk('rm_pca', 'ph', 0.25, **RF_kwargs)

ss_f.do_sk('nir_cdf', 'ph', 0.25, **RF_kwargs)
ss_f.do_sk('nir_pca', 'ph', 0.25, **RF_kwargs)

In [ ]:
lasso_kwargs = {'model':Lasso(), 
                'param_grid':{'alpha':[0.001, 0.01,0.005, 0.05, 0.1,0.2,0.5,0.9],
                            'fit_intercept':[True]
                    },
                'refit':True,
                'metric':mean_squared_error}

In [ ]:
ss_f.do_sk(attr, 'ph', 0.25, **lasso_kwargs)

In [ ]:
from sklearn.metrics import root_mean_squared_error

In [ ]:
rmse_df = {'RandomForestRegressor':{}}
for attr in [attr + '_RandomForestRegressor' for attr in ss_f.best_fits['RandomForestRegressor'].keys()]: 
    ypred = getattr(ss_f, attr)['ypred']
    rmse_df['RandomForestRegressor'][attr.replace('_RandomForestRegressor', '')] = root_mean_squared_error(ss_f.ph_test, ypred) * 100

In [ ]:
sns.heatmap(pd.DataFrame(rmse_df))

In [ ]:
for attr in [f.replace('_ph', '') for f in top_feats]:
    if 'trans' not in attr: 
        ss_f.do_LCEN(attr, 'ph', 0.25, **lcen_kwargs)

In [ ]:
import seaborn as sns 
sns.heatmap(pd.DataFrame(ss_f.best_fits))

In [ ]:
top = 4
top_feats = pd.DataFrame(ss_f.best_fits).sort_values(by='RandomForestRegressor')[:top].index

In [ ]:
f, axs = plt.subplots(2,2, figsize=(10,10), constrained_layout=True)
axs = axs.ravel()
for i, attr in enumerate(top_feats):
    rf_importances = getattr(ss_f, attr + '_RandomForestRegressor')['P'][0].best_estimator_.feature_importances_
    axs[i].plot(rf_importances / np.max(rf_importances))
    axs[i].set_title(attr)
axs[2].plot(ss_f.nir[0] / np.max(ss_f.nir[0]))

In [ ]:
truth_dir = '/Users/amalyajohnson/Desktop/SpectraScope/Datasets/Grape-Chemical-Composition-main/Ground Truth/'
white = 'White Variety.xlsx'
red = 'Red Variety.xlsx'

white = pd.read_excel(truth_dir + white)
red = pd.read_excel(truth_dir + red)

In [ ]:
wl = np.array(data2[name]['wvlength'])

for key in data2[name]['spec']: 
    arr = data2[name]['spec'][key]
    arr = np.array(arr)
    plt.plot(wl, arr[0], label=f'{key}')
    
plt.legend

In [ ]:
shift = data['RAMAN_Data_09082022']['raman_shift']
wl = data['RAMAN_Data_09082022']['wvlength']

shift = np.array(shift)
wl = np.array(wl)

for key in data['RAMAN_Data_09082022']['raman']: 
    for key2 in data['RAMAN_Data_09082022']['raman'][key]: 
        arr = data['RAMAN_Data_09082022']['raman'][key][key2]
        arr = np.array(arr)
        plt.plot(shift, arr[1], label=f'{key} {key2}')
        
plt.legend()